# **Start Section:**

In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/concrete_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d05_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # concrete train
    shutil.copy(glob.glob("/kaggle/input/**/d05_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # concrete test
    print("Kaggle: concrete data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: concrete data copied to ./drive/MyDrive/Regression/concrete_uncertainty_analysis/data


In [2]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks, pinned to the Colab versions,
# and do NOT restart the kernel with os._exit(0).
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 mapie==0.6.0 puncc==0.8.0 properscoring==0.1 XlsxWriter==3.2.9
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install mapie==0.6.0 -q
    !pip install puncc==0.8.0 -q
    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ngboost
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from optuna.samplers import BaseSampler
from optuna.samplers import GridSampler
from optuna.samplers import TPESampler
from optuna.samplers import PartialFixedSampler
from optuna.samplers import CmaEsSampler
from optuna.samplers import QMCSampler
from optuna.samplers import NSGAIIISampler
from optuna.samplers import NSGAIISampler
from optuna.samplers import BruteForceSampler
from optuna.samplers import GPSampler
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from openpyxl.drawing.image import Image
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from mapie.subsample import Subsample
from mapie.regression import MapieRegressor
from deel.puncc.metrics import regression_sharpness, regression_mean_coverage
from deel.puncc.api.prediction import BasePredictor, DualPredictor
from deel.puncc.regression import SplitCP, CVPlus, CQR
from deel.puncc.plotting import plot_prediction_intervals
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from mapie.metrics import regression_coverage_score
from sklearn.model_selection import KFold
from PIL import Image as PImage
from pytorch_tabnet import TabNetRegressor

[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


I0000 00:00:1795359338.584528      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1795359356.164640      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [4]:
# Go to find & replace button and replace (concrete_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.

In [5]:
feature_names = ['C', 'mp', 'FA', 'CA', 'F', 'W_P', 'Adm']
TARGET_LABEL = 'Compressive strength (str)'  # FIX B12: target name used in plot labels (was 'str', the concrete dataset's target, on every dataset)

In [6]:
train_data_path = "./drive/MyDrive/Regression/concrete_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/concrete_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [7]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (189, 8)
First 5 rows of training data:
         C    mp     FA      CA       F       W_P    Adm    str
0  280.80  70.2  858.0  1183.0    0.00  0.450000  0.610  21.56
1  372.15   0.0  975.0   525.0   52.85  0.493081  7.000  34.00
2  360.00  75.0  975.0   525.0   40.00  0.509722  7.000  28.00
3  364.30   0.0  975.0   525.0  110.40  0.503706  7.000  42.00
4  315.00  31.5  780.0  1110.0    0.00  0.370000  5.355  25.82

Shape of test data: (95, 8)
First 5 rows of test data:
         C     mp     FA     CA      F       W_P   Adm   str
0  364.30    0.0  975.0  525.0  60.40  0.503706   7.0  34.0
1  344.30   75.0  975.0  525.0  55.40  0.532965   7.0  36.0
2  390.00    0.0  975.0  525.0  60.00  0.470513   7.0  36.0
3  352.15   75.0  975.0  525.0  47.85  0.521085   7.0  35.0
4  400.00  160.0  801.0  801.0  40.00  0.300000  10.3  44.6


In [8]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
# FIX D4 / D5 (approved 2026-11-22): settings that follow the test-set size
N_BINS = max(2, min(10, len(y_test) // 10))  # D4: bins for binned metrics (was 10 on every dataset; pile: 2-3 points per bin)
COP_T_BURNIN = min(50, len(y_test) // 2)  # D5: COP burn-in (was 50, so with < 50 test points COP's correction never started)
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (189, 7)
Shape of y_train: (189,)
Shape of X_test: (95, 7)
Shape of y_test: (95,)


In [9]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[-1.56879184  0.95792014 -0.09685034  1.33138508 -0.85369002  0.16810471
  -0.08799752]
 [ 0.40300047 -0.7708921   0.73537613 -0.93459372  0.061075    0.78285857
  -0.0664621 ]
 [ 0.14074238  1.07612953  0.73537613 -0.93459372 -0.16134185  1.020321
  -0.0664621 ]
 [ 0.233558   -0.7708921   0.73537613 -0.93459372  1.05719093  0.93447436
  -0.0664621 ]
 [-0.83058388  0.00485698 -0.65166799  1.07999229 -0.85369002 -0.97347298
  -0.07200604]]

First five rows of normalized X_test:
[[ 0.233558   -0.7708921   0.73537613 -0.93459372  0.19175572  0.93447436
  -0.0664621 ]
 [-0.19814256  1.07612953  0.73537613 -0.93459372  0.1052122   1.35199213
  -0.0664621 ]
 [ 0.78829322 -0.7708921   0.73537613 -0.93459372  0.18483223  0.4608195
  -0.0664621 ]
 [-0.02870009  1.07612953  0.73537613 -0.93459372 -0.02546852  1.18246784
  -0.0664621 ]
 [ 1.0041435   3.1694207  -0.50229401  0.01587763 -0.16134185 -1.97235346
  -0.05534052]]


# **Functions:**

In [10]:
# Define the model classes
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}

def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

In [11]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost',
        'GradientBoosting': 'Gradient Boosting',
        'HGBR' : 'HistGradientBoosting',
        'CatBoost' : 'CatBoost',
        'TabNet' : 'TabNet',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']

In [12]:
# FIX Q2: quantile crossing. Each quantile level is a separate model, so for some rows the
# predictions can come out in the wrong order (e.g. q0.10 < q0.05). Before any metric we sort
# each row's quantile predictions (monotone rearrangement, Chernozhukov et al. 2010) and record
# how many rows crossed, for the Excel report.
quantile_crossing_report = {}

def rearrange_quantiles(predictions_df, tag=None):
    q_cols = sorted(c for c in predictions_df.columns if c != 'Actual')
    values = predictions_df[q_cols].to_numpy(dtype=float)
    crossed = (np.diff(values, axis=1) < 0).any(axis=1)
    if tag is not None:
        quantile_crossing_report[tag] = {'rows': int(len(values)), 'rows_crossing': int(crossed.sum()),
                                         'pct_crossing': round(100.0 * crossed.mean(), 2)}
    out = predictions_df.copy()
    out[q_cols] = np.sort(values, axis=1)
    return out


In [13]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def quantile_regression_print(predictions_df, file_path):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    # Extract actual values
    actual_values = predictions_df['Actual'].values

    # Debugging: Print the first few actual values
    print("First few actual values:", actual_values[:5])

    # Calculate coverage for the specific interval (0.05 to 0.95)
    lower_quantile = 0.05
    median_quantile = 0.5
    upper_quantile = 0.95

    lower_preds = predictions_df[lower_quantile].values
    median_preds = predictions_df[median_quantile].values
    upper_preds = predictions_df[upper_quantile].values

    in_interval = ((actual_values >= lower_preds) & (actual_values <= upper_preds))
    coverage = in_interval.mean()
    print(f"Coverage of 90% prediction interval: {coverage * 100:.2f}%")

    # Create a new Excel workbook and add a worksheet
    wb = Workbook()
    ws = wb.active
    ws.title = "quantile_regression_print"

    # Plot actual vs. all predicted quantiles
    num_points = len(predictions_df)
    indices = np.arange(num_points)

    plt.figure(figsize=(12, 6))
    plt.plot(indices, actual_values[:num_points], label=f'Actual {TARGET_LABEL}', marker='o', linestyle='-', color='black')

    # Loop over all columns assumed to be quantile predictions
    for column in predictions_df.columns:
        if column != 'Actual':  # Skip the actual values column
            plt.plot(indices, predictions_df[column][:num_points],
                     label=f'Predicted Quantile {column}', linestyle='--')

    plt.xlabel('Sample Number')
    plt.ylabel(f'{TARGET_LABEL}')
    plt.title(f'Actual vs. Predicted Quantile {TARGET_LABEL}')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)

    # Save the plot to a BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Insert the image into the Excel sheet
    img = Image(img_data)
    ws.add_image(img, 'A1')

    # Plot actual values and specific prediction intervals (0.05, 0.5, 0.95)
    plt.figure(figsize=(12, 6))
    plt.plot(indices, actual_values, label=f'Actual {TARGET_LABEL}', marker='o', linestyle='-', color='black')  # Connect points with a line
    plt.plot(indices, median_preds, label=f'Median Prediction ({median_quantile})', marker='x', linestyle='-', color='blue')
    plt.fill_between(
        indices,
        lower_preds,
        upper_preds,
        color='blue',
        alpha=0.5,
        label=f'Prediction Interval ({lower_quantile}-{upper_quantile})'
    )
    plt.xlabel('Sample Number')
    plt.ylabel(f' {TARGET_LABEL}')
    plt.title(f'Actual {TARGET_LABEL} with Prediction Intervals')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1, 1))

    # Save the second plot to another BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Insert the second image into the Excel sheet
    img = Image(img_data)
    ws.add_image(img, 'A30')  # Adjust the cell position as needed

    # Save the workbook
    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

In [14]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def evaluation_features_print(predictions_df, X_test, file_path):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    # Ensure X_test is a DataFrame
    if isinstance(X_test, np.ndarray):
        X_test = pd.DataFrame(X_test, columns=feature_names)
    X_test.columns = feature_names

    # Get quantile columns from predictions_df except 'Actual'
    quantile_columns = [col for col in predictions_df.columns if col != 'Actual']

    # Check if the Excel file already exists
    if os.path.exists(file_path):
        wb = load_workbook(_ensure_excel_file(file_path))
        ws = wb.create_sheet(title="New Evaluation Features")
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = "evaluation_features_print"

    image_row = 1  # Where to start images

    for feature_to_plot in feature_names:
        feature_values = X_test[feature_to_plot]
        plot_df = predictions_df.copy()
        plot_df[feature_to_plot] = feature_values.values

        # Sort by current feature to plot lines well
        plot_df = plot_df.sort_values(by=feature_to_plot)

        # Begin Matplotlib plot
        plt.figure(figsize=(6, 4))
        # Plot each quantile as a line
        for quantile in quantile_columns:
            plt.plot(plot_df[feature_to_plot], plot_df[quantile], label=f"{quantile}")

        # Plot actual values as scatter
        plt.scatter(plot_df[feature_to_plot], plot_df['Actual'], color='black', s=12, label=f"Actual {TARGET_LABEL}")

        plt.title(f"Predicted Quantiles vs. {feature_to_plot}")
        plt.xlabel(feature_to_plot)
        plt.ylabel(f"{TARGET_LABEL}")
        plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
        plt.tight_layout()

        # Save figure to a BytesIO stream
        img_bytes = BytesIO()
        img_bytes
        plt.savefig(img_bytes, format="png")
        plt.close()
        img_bytes.seek(0)

        img = Image(img_bytes)
        img.anchor = f"A{image_row}"
        ws.add_image(img)

        image_row += 20  # Adjust as you like

    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

In [15]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def evaluate_uncertainity_matrix(predictions_df, quantiles, excel_file_path, model_name='Model'):
    predictions_df = rearrange_quantiles(predictions_df, excel_file_path)  # FIX Q2

   # Extract the actual target values
    y_true = predictions_df['Actual']

    # Open existing Excel file
    wb = load_workbook(_ensure_excel_file(excel_file_path))

    # Create unique sheet names
    def get_unique_sheet_name(wb, base_name):
        sheet_name = base_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base_name}_{i}"
            i += 1
        return sheet_name

    # Create separate sheets for plots and values
    plots_sheet_title = get_unique_sheet_name(wb, 'Plots')
    values_sheet_title = get_unique_sheet_name(wb, 'Values')

    ws_plots = wb.create_sheet(title=plots_sheet_title)
    ws_values = wb.create_sheet(title=values_sheet_title)
    # FIX Q2: how often the raw quantile predictions crossed (before sorting)
    _cr = quantile_crossing_report.get(excel_file_path, {})
    ws_values.append(['Rows with quantile crossing (before sorting)', _cr.get('rows_crossing'), 'of', _cr.get('rows'), 'rows', f"{_cr.get('pct_crossing')}%"])
    print(f"Quantile crossing before sorting: {_cr.get('rows_crossing')} of {_cr.get('rows')} rows ({_cr.get('pct_crossing')}%)")

    # Positions to place images
    image_positions = ['A1', 'A25', 'A49', 'A73']  # Adjust as needed

    ### 1. Validity / Coverage ###

    # Compute empirical coverage for each quantile
    empirical_quantiles = []
    for q in quantiles:
        coverage = (predictions_df[q] >= y_true).mean()
        empirical_quantiles.append(coverage)

    # Plot theoretical quantiles vs empirical quantiles
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=quantiles, y=quantiles, color="magenta", linestyle='--', linewidth=2, label="Ideal")
    sns.lineplot(x=quantiles, y=empirical_quantiles, color="blue", linewidth=2, label="Empirical")
    sns.scatterplot(x=quantiles, y=empirical_quantiles, color="blue", s=50)
    plt.xlabel("Nominal Quantile Levels")
    plt.ylabel("Empirical Quantile Levels")
    plt.title(f"Validity / Coverage - {model_name}")
    plt.legend()

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add the image to the plots worksheet
    ws_plots.add_image(img, image_positions[0])

    ### 2. Sharpness / Interval Length ###

    # Define coverage levels to evaluate
    coverage_levels = [0.6, 0.8, 0.9]
    average_interval_lengths = []

    # Compute average interval lengths for each coverage level
    for c in coverage_levels:
        lower_q = (1 - c) / 2
        upper_q = 1 - lower_q
        # Find the nearest quantiles in our quantiles list
        lower_quantile = min(quantiles, key=lambda x: abs(x - lower_q))
        upper_quantile = min(quantiles, key=lambda x: abs(x - upper_q))

        # Compute interval length for each observation
        interval_length = predictions_df[upper_quantile] - predictions_df[lower_quantile]
        avg_interval_length = interval_length.mean()
        average_interval_lengths.append(avg_interval_length)
        # Write interval length information to the values worksheet
        ws_values.append([f'Coverage level: {c*100:.0f}%', f'Interval: [{lower_quantile}, {upper_quantile}]', f'Average Interval Length: {avg_interval_length:.4f}'])

    # Plot coverage levels vs average interval lengths
    plt.figure(figsize=(10, 6))
    plt.plot([c*100 for c in coverage_levels], average_interval_lengths, marker='o')
    plt.xlabel('Coverage Level (%)')
    plt.ylabel('Average Interval Length')
    plt.title(f'Sharpness / Interval Length - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[1])

    ### 3. Negative Log-Likelihood (NLL) ###

    # Use the 5th and 95th percentiles to estimate standard deviation
    if 0.05 in quantiles and 0.95 in quantiles:
        z_lower = norm.ppf(0.05)  # z-score for 5% quantile (~ -1.6449)
        z_upper = norm.ppf(0.95)  # z-score for 95% quantile (~ +1.6449)

        # Extract quantile predictions
        q_lower = predictions_df[0.05]
        q_upper = predictions_df[0.95]

    elif 0.1 in quantiles and 0.9 in quantiles:
        z_lower = norm.ppf(0.1)  # z-score for 10% quantile (~ -1.2816)
        z_upper = norm.ppf(0.9)  # z-score for 90% quantile (~ +1.2816)

        # Extract quantile predictions
        q_lower = predictions_df[0.1]
        q_upper = predictions_df[0.9]
    else:
        raise ValueError("Required quantiles for NLL estimation not found in quantiles list.")

    # Use median as the mean estimate
    if 0.5 in quantiles:
        q_median = predictions_df[0.5]
    else:
        # If 0.5 quantile is not available, use the middle quantile
        q_median = predictions_df[quantiles[len(quantiles)/2]]

    # Estimate standard deviation for each observation
    std_estimates = (q_upper - q_lower) / (z_upper - z_lower)

    # Ensure standard deviations are positive and non-zero
    std_estimates = std_estimates.clip(lower=1e-6)

    # Extract mean estimates (median predictions)
    mean_estimates = q_median

    # Compute Negative Log-Likelihood for each observation
    nll = -norm.logpdf(y_true, loc=mean_estimates, scale=std_estimates)

    # Compute average NLL
    average_nll = nll.mean()
    # Write average NLL to the values worksheet
    ws_values.append(['Average Negative Log-Likelihood (NLL)', average_nll])

    # Write NLL values to the values worksheet
    ws_values.append(['NLL Values'])
    for value in nll:
        ws_values.append([value])

    # Plot ECDF of NLL values
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(nll, color='blue', linewidth=2)
    plt.xlabel('Negative Log-Likelihood (NLL)')
    plt.ylabel('ECDF')
    plt.title(f'NLL ECDF - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[2])

    ### 4. Continuous Ranked Probability Score (CRPS) ###

    # Prepare ensemble of quantile predictions for each observation
    ensemble_predictions = predictions_df[quantiles].to_numpy()

    # Ensure quantile predictions are sorted for each observation
    ensemble_predictions.sort(axis=1)

    # Compute CRPS for each observation using ensemble predictions
    # FIX Q4: CRPS from a set of quantiles = 2 * integral of the pinball loss over tau.
    # Each quantile level gets a weight equal to the share of [0, 1] it represents
    # (levels are unevenly spaced). Was: ps.crps_ensemble, which treats the 13 quantiles as
    # 13 equally likely samples.
    _taus = np.array(quantiles, dtype=float)
    _edges = np.concatenate([[0.0], (_taus[1:] + _taus[:-1]) / 2.0, [1.0]])
    _w = np.diff(_edges)
    _err = np.asarray(y_true, dtype=float)[:, None] - ensemble_predictions
    _pinball = np.maximum(_taus * _err, (_taus - 1.0) * _err)
    crps_values = 2.0 * (_pinball * _w).sum(axis=1)

    # Compute average CRPS
    average_crps = crps_values.mean()
    # Write average CRPS to the values worksheet
    ws_values.append(['Average Continuous Ranked Probability Score (CRPS)', average_crps])

    # Write CRPS values to the values worksheet
    ws_values.append(['CRPS Values'])
    for value in crps_values:
        ws_values.append([value])

    # Plot ECDF of CRPS values
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(crps_values, color='green', linewidth=2)
    plt.xlabel('Continuous Ranked Probability Score (CRPS)')
    plt.ylabel('ECDF')
    plt.title(f'CRPS ECDF - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[3])

    # Save the workbook
    _ensure_parent_dir(excel_file_path)
    wb.save(_ensure_parent_dir(excel_file_path))

    # Return a dictionary of results
    results = {
        'empirical_quantiles': empirical_quantiles,
        'average_interval_lengths': average_interval_lengths,
        'coverage_levels': coverage_levels,
        'average_nll': average_nll,
        'average_crps': average_crps,
        'nll_values': nll,
        'crps_values': crps_values
    }

    return results

In [16]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def calibrate_and_plot_intervals(predictions_df, excel_file_path, alpha=0.1, max_samples=1000, random_seed=42):
    predictions_df = rearrange_quantiles(predictions_df, excel_file_path)  # FIX Q2
    # Extract necessary columns
    lower = predictions_df[0.05].fillna(0)
    upper = predictions_df[0.95].fillna(0)
    pred = predictions_df[0.5].fillna(0)
    labels = predictions_df['Actual'].fillna(0)

    # Problem setup
    total_samples = labels.shape[0]
    n = min(max_samples, total_samples)  # Ensure n does not exceed the number of samples

    # Ensure there are enough samples for both calibration and validation
    if total_samples <= n:
        n = total_samples // 2

    # Split the data into calibration and validation sets
    np.random.seed(random_seed)  # Set the random seed for reproducibility
    idx = np.array([1] * n + [0] * (total_samples - n)) > 0
    np.random.shuffle(idx)
    cal_labels, val_labels = labels[idx], labels[~idx]
    cal_upper, val_upper = upper[idx], upper[~idx]
    cal_lower, val_lower = lower[idx], lower[~idx]
    cal_pred, val_pred = pred[idx], pred[~idx]

    # Calculate uncertainty intervals
    cal_U = cal_upper - cal_lower
    val_U = val_upper - val_lower

    # Avoid division by zero by replacing zero intervals with a small number
    cal_U[cal_U == 0] = np.finfo(float).eps
    val_U[val_U == 0] = np.finfo(float).eps

    # Get scores
    cal_scores = np.abs(cal_pred - cal_labels) / cal_U

    # Debugging: Check for NaN in scores
    if np.isnan(cal_scores).any():
        print("NaN values found in cal_scores. Check data integrity.")

    # Get the score quantile
    qhat = np.quantile(cal_scores, np.ceil((n + 1) * (1 - alpha)) / n, interpolation='higher')

    # Deploy (output=lower and upper adjusted quantiles)
    prediction_sets = [val_pred - val_U * qhat, val_pred + val_U * qhat]

    # Calculate empirical coverage (before and after calibration)
    prediction_sets_uncalibrated = [val_lower, val_upper]
    empirical_coverage_uncalibrated = ((val_labels >= prediction_sets_uncalibrated[0]) & (val_labels <= prediction_sets_uncalibrated[1])).mean() * 100
    print(f"The empirical coverage before calibration is: {empirical_coverage_uncalibrated:.2f}%")

    empirical_coverage = ((val_labels >= prediction_sets[0]) & (val_labels <= prediction_sets[1])).mean() * 100
    print(f"The empirical coverage after calibration is: {empirical_coverage:.2f}%")

    # Create a DataFrame with all the necessary data
    data = pd.DataFrame({
        'Index': np.arange(len(val_labels)),
        'Uncalibrated Lower': val_lower,
        'Uncalibrated Upper': val_upper,
        'Calibrated Lower': prediction_sets[0],
        'Calibrated Upper': prediction_sets[1],
        'True Label': val_labels,
        'Predicted Value': val_pred
    })

    plt.figure(figsize=(14, 8))

    # Plot the uncalibrated prediction intervals as a shaded area
    plt.fill_between(
        data['Index'],
        data['Uncalibrated Lower'],
        data['Uncalibrated Upper'],
        color='blue',
        alpha=0.5,
        label='Uncalibrated Prediction Interval'
    )

    # Plot the calibrated prediction intervals as a shaded area
    plt.fill_between(
        data['Index'],
        data['Calibrated Lower'],
        data['Calibrated Upper'],
        color='lightgreen',
        alpha=0.5,
        label='Calibrated Prediction Interval'
    )

    # Plot the true labels as points and connect them with a line
    plt.plot(
        data['Index'],
        data['True Label'],
        'o-',
        color='red',
        markersize=4,
        label='Actual Value'
    )

    # Plot the predicted values as points and connect them with a line
    plt.plot(
        data['Index'],
        data['Predicted Value'],
        'o-',
        color='black',
        markersize=4,
        label='Predicted Value'
    )

    plt.xlabel('Sample Number')
    plt.ylabel(f'{TARGET_LABEL}')
    plt.title('Calibrated and Uncalibrated Prediction Intervals with True and Predicted Values')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()

    # Save the plot as an image
    image_path = 'plot.png'
    _ensure_parent_dir(image_path)
    plt.savefig(_ensure_parent_dir(image_path))
    plt.close()

    # Load the existing Excel file
    workbook = load_workbook(_ensure_excel_file(excel_file_path))

    # Create a new sheet for the plot
    new_sheet_name = "calibrate_and_plot_intervals"
    workbook.create_sheet(title=new_sheet_name)

    # Insert the image into the new sheet
    sheet = workbook[new_sheet_name]
    img = Image(image_path)
    sheet.add_image(img, 'A1')

    # Save the workbook
    _ensure_parent_dir(excel_file_path)
    workbook.save(_ensure_parent_dir(excel_file_path))

In [17]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def plot_quantile_intervals_to_excel(predictions_df, file_path, alpha=0.1):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    """
    Plots conformal prediction intervals and their coverage, and saves the plots to an Excel file.

    Parameters:
    - predictions_df: DataFrame containing prediction quantiles and actual values.
    - file_path: Path to the Excel file where plots will be saved.
    - alpha: Desired coverage level (default is 0.1).
    """
    # Extract mean prediction and uncertainty
    mean_prediction = predictions_df[0.5]
    uncertainty = predictions_df[0.95] - predictions_df[0.05]
    actual_values = predictions_df['Actual']

    # Calculate conformal scores
    conformal_scores = np.abs(mean_prediction - actual_values) / uncertainty

    # Function to find weighted quantile
    def weighted_quantile(values, quantile, sample_weight=None):
        values = np.array(values)
        if sample_weight is None:
            sample_weight = np.ones(len(values))
        sorter = np.argsort(values)
        values, sample_weight = values[sorter], sample_weight[sorter]
        weighted_quantiles = np.cumsum(sample_weight) - 0.5 * sample_weight
        weighted_quantiles /= np.sum(sample_weight)
        return np.interp(quantile, weighted_quantiles, values)

    # Calculate weighted quantile
    weights = np.ones_like(conformal_scores)  # Uniform weights for simplicity
    quantile = weighted_quantile(conformal_scores, 1 - alpha, sample_weight=weights)

    # Calculate prediction intervals
    lower_bound = mean_prediction - quantile * uncertainty
    upper_bound = mean_prediction + quantile * uncertainty

    # Naive conformal prediction
    naive_quantile = np.quantile(conformal_scores, 1 - alpha)
    naive_lower_bound = mean_prediction - naive_quantile * uncertainty
    naive_upper_bound = mean_prediction + naive_quantile * uncertainty

    # Coverage calculation (not plotted, just for potential diagnostics)
    weighted_coverage_points = (actual_values >= lower_bound) & (actual_values <= upper_bound)
    naive_coverage_points = (actual_values >= naive_lower_bound) & (actual_values <= naive_upper_bound)

    # Visualization using Matplotlib
    fig, ax = plt.subplots(figsize=(10, 5))

    x = np.arange(len(predictions_df))
    # Actual Values
    ax.plot(x, actual_values, label='Actual', color='black', linewidth=2)

    # Mean Prediction
    ax.plot(x, mean_prediction, label='Mean Prediction', color='darkorange', linewidth=2)

    # Weighted Prediction Interval (blue fill)
    ax.fill_between(x, lower_bound, upper_bound, color='blue', alpha=0.3, label='Weighted Interval')
    # Optionally, plot the bounds as lines if you want:
    # ax.plot(x, lower_bound, color='blue', linewidth=1, linestyle='--')
    # ax.plot(x, upper_bound, color='blue', linewidth=1, linestyle='--')

    # Naive Prediction Interval (red fill)
    ax.fill_between(x, naive_lower_bound, naive_upper_bound, color='red', alpha=0.3, label='Naive Interval')
    # Optionally, plot the bounds as lines:
    # ax.plot(x, naive_lower_bound, color='red', linewidth=1, linestyle='--')
    # ax.plot(x, naive_upper_bound, color='red', linewidth=1, linestyle='--')

    ax.set_title('Quantile Prediction Intervals')
    ax.set_xlabel('Sample Number')
    ax.set_ylabel(f'{TARGET_LABEL}')
    ax.grid(False)
    ax.set_facecolor('white')

    # Place the legend outside the plot
    ax.legend(loc='center left', bbox_to_anchor=(1.05, 0.5), borderaxespad=0.0)

    plt.tight_layout(rect=[0,0,0.80,1.0])  # Make space for legend

    # Save the plot to a BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Add image to Excel
    if os.path.exists(file_path):
        wb = load_workbook(_ensure_excel_file(file_path))
        ws = wb.create_sheet(title="plot_quantile_intervals_to_excel")
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = "plot_quantile_intervals_to_excel"

    img = Image(img_data)
    img.anchor = 'A1'
    ws.add_image(img)
    ws.column_dimensions['A'].width = 30
    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

# **Tuned Parameters**

In [18]:
# FIX K2: load the tuning results saved by hyperparameter_tuning/Optuna_autosampler.ipynb
# (was: a hand-pasted copy of an older tuning run that had been selected on the TEST set).
import pickle
if IN_KAGGLE:
    _bp = glob.glob("/kaggle/input/**/hyperparameter_tuning/best_params.pkl", recursive=True)[0]
else:
    _bp = "./drive/MyDrive/Regression/concrete_uncertainty_analysis/hyperparameter_tuning/best_params.pkl"
with open(_bp, "rb") as f:
    best_scores_autosampler = pickle.load(f)
print("Loaded tuning results from", _bp, "-", len(best_scores_autosampler), "model x pruner entries")


Loaded tuning results from /kaggle/input/datasets/daneshjangra/concrete-tuned-params/hyperparameter_tuning/best_params.pkl - 80 model x pruner entries


In [19]:
best_scores_autosampler

{('Random Forest', 'MedianPruner'): {'best_score': 19.61800683683138,
  'cv_rmse': 4.429221922282895,
  'best_params': {'n_estimators': 500,
   'criterion': 'friedman_mse',
   'max_depth': 30,
   'min_samples_split': 0.01,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 'log2',
   'max_leaf_nodes': None,
   'min_impurity_decrease': 0.0,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_alpha': 0.001},
  'test_mse': 11.477140958909033,
  'test_rmse': np.float64(3.3877929332987624),
  'test_corr_coef': np.float64(0.9430872254495873),
  'pruner': 'MedianPruner'},
 ('Random Forest', 'NopPruner'): {'best_score': 19.61800683683138,
  'cv_rmse': 4.429221922282895,
  'best_params': {'n_estimators': 500,
   'criterion': 'friedman_mse',
   'max_depth': 30,
   'min_samples_split': 0.01,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 'log2',
   'max_leaf_nodes': None,
   'min_impurity_decrease

# **Quantile Regression with Lightgbm**

In [20]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_LGBM_df = pd.DataFrame()


# Train a model for each quantile and make predictions using LightGBM
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Update params with quantile-specific settings
    params.update({
        'objective': 'quantile',
        'alpha': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = LGBMRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_LGBM_df[q] = predictions

# Add actual target values to the DataFrame
predictions_LGBM_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_LGBM_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  25.922896  26.388425  27.751919  32.433610  33.007925  33.009659   
1  25.651126  31.200928  32.704197  34.481275  34.934801  36.414487   
2  30.938161  32.494963  32.911354  34.570882  32.794408  34.435775   
3  26.312107  31.859629  32.875124  34.221650  34.770330  37.223855   
4  43.360197  40.106591  42.977236  38.342269  44.552150  40.890906   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  32.999468  32.668038  32.811037  34.453255  35.781507  36.937625   
1  36.328580  37.011580  37.132992  37.158416  36.645178  36.908751   
2  34.574644  35.015921  34.825912  34.991745  36.995246  37.867159   
3  36.624199  36.689904  37.458191  37.131736  36.397378  37.043054   
4  44.751812  38.835099  47.979142  45.804553  49.259375  49.439518   

        0.95  Actual  
0  40.378646    34.0  
1  36.784795    36.0  
2  39.711925    36.0  
3  37.656957    35.0  
4  48.297310    44.6  


In [21]:
quantile_regression_print(predictions_LGBM_df,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

First few actual values: [34.  36.  36.  35.  44.6]
Coverage of 90% prediction interval: 78.95%


In [22]:
evaluation_features_print(predictions_LGBM_df,x_test,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

In [23]:
evaluate_uncertainity_matrix(predictions_LGBM_df, quantiles, "./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/LightGBM.xlsx", model_name='LightGBM')

Quantile crossing before sorting: 93 of 95 rows (97.89%)


{'empirical_quantiles': [np.float64(0.09473684210526316),
  np.float64(0.1368421052631579),
  np.float64(0.23157894736842105),
  np.float64(0.3368421052631579),
  np.float64(0.3684210526315789),
  np.float64(0.45263157894736844),
  np.float64(0.49473684210526314),
  np.float64(0.5684210526315789),
  np.float64(0.6631578947368421),
  np.float64(0.7052631578947368),
  np.float64(0.7684210526315789),
  np.float64(0.8),
  np.float64(0.8842105263157894)],
 'average_interval_lengths': [np.float64(3.293232275372713),
  np.float64(6.519743707497124),
  np.float64(10.352474693857488)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(187.85973796134874),
 'average_crps': np.float64(1.7743654997036897),
 'nll_values': array([2.42515399e+00, 2.17814116e+00, 2.04271615e+00, 2.26781253e+00,
        2.13493777e+00, 1.82504185e+00, 6.19932229e+00, 1.55400039e+00,
        3.04645060e+00, 2.32080066e+00, 8.45431867e-01, 1.57373210e+00,
        1.77636967e+00, 2.54731521e+00, 3.20028197e+

In [24]:
calibrate_and_plot_intervals(predictions_LGBM_df, "./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

The empirical coverage before calibration is: 79.17%
The empirical coverage after calibration is: 87.50%


In [25]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_LGBM_df,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

# **Quantile Regression with XGBoost**

In [26]:
# Get the best hyperparameters for XGBRegressor
best_paramst = get_best_model_params(best_scores_autosampler, 'XGBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_XGB_df = pd.DataFrame()

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_paramst.copy()  # FIX Q1: was best_params (= LightGBM's settings from the cell above)

    # Adjust according to your XGBoost version
    params.update({
        'objective': 'reg:quantileerror',  # Use the correct objective
        'quantile_alpha': q,                         # Use 'alpha' to set the quantile level
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = XGBRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_XGB_df[q] = predictions

# Add actual target values to the DataFrame
predictions_XGB_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_XGB_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  24.573736  26.971348  28.720428  29.138767  31.923574  32.253395   
1  24.573736  24.497898  25.038534  24.651608  27.814920  30.395464   
2  24.573736  27.636446  30.245193  32.439579  34.844391  35.485840   
3  24.573736  24.497898  25.297384  24.751394  27.814920  30.439539   
4  24.573736  27.636446  30.504044  33.297935  38.048141  40.979992   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  32.987991  37.382221  37.979916  40.405739  40.406647  43.456779   
1  33.636742  33.324348  34.822212  35.677547  38.815163  42.982956   
2  35.775848  37.898018  39.422527  41.540752  42.529190  45.438805   
3  33.636742  33.324348  34.822212  35.677547  38.815163  42.982956   
4  45.858559  48.246162  50.617161  54.297058  54.866074  55.444870   

        0.95  Actual  
0  55.763855    34.0  
1  55.763855    36.0  
2  55.763855    36.0  
3  55.763855    35.0  
4  58.230606    44.6  


In [27]:
quantile_regression_print(predictions_XGB_df,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

First few actual values: [34.  36.  36.  35.  44.6]
Coverage of 90% prediction interval: 90.53%


In [28]:
evaluation_features_print(predictions_XGB_df,x_test,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

In [29]:
evaluate_uncertainity_matrix(predictions_XGB_df, quantiles, "./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/XGBoost.xlsx", model_name='XGBoost')

Quantile crossing before sorting: 75 of 95 rows (78.95%)


{'empirical_quantiles': [np.float64(0.031578947368421054),
  np.float64(0.06315789473684211),
  np.float64(0.09473684210526316),
  np.float64(0.14736842105263157),
  np.float64(0.24210526315789474),
  np.float64(0.29473684210526313),
  np.float64(0.43157894736842106),
  np.float64(0.5578947368421052),
  np.float64(0.6210526315789474),
  np.float64(0.7894736842105263),
  np.float64(0.8526315789473684),
  np.float64(0.9052631578947369),
  np.float64(0.9368421052631579)],
 'average_interval_lengths': [np.float64(9.681855312146638),
  np.float64(19.263894613165604),
  np.float64(31.668090619538958)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(3.2990948516365766),
 'average_crps': np.float64(3.0012260687271923),
 'nll_values': array([3.17393793, 3.21029764, 3.16852076, 3.18621188, 3.25192345,
        3.1929747 , 3.81564748, 3.17012391, 3.66747248, 3.16892205,
        3.16913332, 3.16838988, 3.18215812, 3.46376136, 3.47000672,
        3.20706374, 3.34672491, 3.17651105, 

In [30]:
calibrate_and_plot_intervals(predictions_XGB_df, "./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

The empirical coverage before calibration is: 93.75%
The empirical coverage after calibration is: 93.75%


In [31]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_XGB_df, "./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

# **Quantile Regression with GPBoost**

In [32]:
# Get the best hyperparameters for GPBoostRegressor
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_GPBoost_df = pd.DataFrame()

# Parameters that may conflict with quantile-specific settings
incompatible_keys = ['objective', 'alpha', 'quantile_alpha', 'boosting_type', 'metric', 'eval_metric']

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Remove incompatible parameters
    for key in incompatible_keys:
        params.pop(key, None)

    # Update params with quantile-specific settings
    params.update({
        'objective': 'quantile',
        'alpha': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = GPBoostRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_GPBoost_df[q] = predictions

# Add actual target values to the DataFrame
predictions_GPBoost_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_GPBoost_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  27.000235  31.845960  31.727309  31.817135  32.818817  32.324274   
1  33.460614  34.134441  32.274102  34.849695  36.071145  34.744518   
2  32.348204  34.346460  32.259682  32.194976  34.099780  34.261204   
3  30.628403  33.946851  33.013688  32.774873  34.821116  34.994069   
4  41.637403  42.494964  43.255150  42.179911  43.173340  41.504113   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  32.214319  32.649957  34.538575  34.403011  36.724519  38.375755   
1  37.188615  35.532007  37.134945  38.308695  36.679685  37.796276   
2  34.653106  34.664373  34.951862  35.151698  36.834878  39.003031   
3  36.490205  35.176515  36.142659  37.146912  34.165758  37.265003   
4  41.010726  40.376492  44.259615  46.117688  40.869506  39.681353   

        0.95  Actual  
0  40.394941    34.0  
1  37.002828    36.0  
2  40.456907    36.0  
3  36.837437    35.0  
4  39.697975    44.6  


In [33]:
quantile_regression_print(predictions_GPBoost_df,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

First few actual values: [34.  36.  36.  35.  44.6]
Coverage of 90% prediction interval: 73.68%


In [34]:
evaluation_features_print(predictions_GPBoost_df,x_test,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

In [35]:
evaluate_uncertainity_matrix(predictions_GPBoost_df, quantiles,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/GPBoost.xlsx", model_name='GPBoost')

Quantile crossing before sorting: 94 of 95 rows (98.95%)


{'empirical_quantiles': [np.float64(0.12631578947368421),
  np.float64(0.16842105263157894),
  np.float64(0.2736842105263158),
  np.float64(0.3157894736842105),
  np.float64(0.35789473684210527),
  np.float64(0.42105263157894735),
  np.float64(0.47368421052631576),
  np.float64(0.5684210526315789),
  np.float64(0.6),
  np.float64(0.6526315789473685),
  np.float64(0.7263157894736842),
  np.float64(0.7578947368421053),
  np.float64(0.8631578947368421)],
 'average_interval_lengths': [np.float64(2.7698751021796153),
  np.float64(5.534636876269618),
  np.float64(8.04821631680244)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(10109.541485743363),
 'average_crps': np.float64(1.7938720012778537),
 'nll_values': array([2.37796799e+00, 1.52640046e+00, 1.98360805e+00, 1.62074407e+00,
        2.73654229e+00, 1.25269631e+00, 9.09701440e+00, 1.20462485e+00,
        2.75391410e+00, 2.18640758e+00, 8.71976628e-01, 5.24836895e-01,
        2.60416859e+00, 3.06981895e+00, 2.88928945e+

In [36]:
calibrate_and_plot_intervals(predictions_GPBoost_df,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

The empirical coverage before calibration is: 70.83%
The empirical coverage after calibration is: 91.67%


In [37]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_GPBoost_df,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

# **Quantile Regression with NGBoost**

In [38]:
# Correct best_params assignment
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')

# DataFrame to store predictions
predictions_NGB_df = pd.DataFrame()

# Remove 'Score' and 'Dist' if present in best_params
for key in ['Score', 'Dist']:
    best_params.pop(key, None)

# Update params with model settings
best_params.update({
    'Dist': Normal,
    'Score': LogScore,
    'random_state': 42,
    'verbose': True
})

# Initialize and fit the NGBoost model once
model = NGBRegressor(**best_params)

# Fit the model
model.fit(X_train, y_train)

# Make predictions
pred_dist = model.pred_dist(X_test)

# Extract the mean (mu) and standard deviation (sigma) of the predicted distributions
mu = pred_dist.loc    # Mean predictions
sigma = pred_dist.scale  # Standard deviation predictions

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# For each quantile, compute predictions
for q in quantiles:
    # Calculate the z-score for the desired quantile
    z = norm.ppf(q)

    # Compute the quantile prediction per sample
    predictions = mu + z * sigma

    # Add predictions to the DataFrame
    predictions_NGB_df[q] = predictions

# Add actual target values to the DataFrame
predictions_NGB_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_NGB_df.head())

[iter 0] loss=3.6975 val_loss=0.0000 scale=1.0000 norm=7.9235


[iter 100] loss=2.4604 val_loss=0.0000 scale=1.0000 norm=2.5097


[iter 200] loss=1.9152 val_loss=0.0000 scale=1.0000 norm=1.6460


[iter 300] loss=1.6151 val_loss=0.0000 scale=1.0000 norm=1.2906


[iter 400] loss=1.4229 val_loss=0.0000 scale=1.0000 norm=1.1137


[iter 500] loss=1.1654 val_loss=0.0000 scale=1.0000 norm=0.9015


[iter 600] loss=1.0868 val_loss=0.0000 scale=0.5000 norm=0.4358


[iter 700] loss=0.8859 val_loss=0.0000 scale=1.0000 norm=0.7479


[iter 800] loss=0.7918 val_loss=0.0000 scale=1.0000 norm=0.7364


[iter 900] loss=0.6772 val_loss=0.0000 scale=1.0000 norm=0.6390


        0.05        0.1       0.15        0.2        0.3        0.4  \
0  31.770896  31.964444  32.095030  32.198816  32.367814  32.512217   
1  35.334998  35.412138  35.464184  35.505548  35.572904  35.630456   
2  32.585624  32.879039  33.077005  33.234342  33.490540  33.709452   
3  35.550645  35.622133  35.670365  35.708698  35.771118  35.824453   
4  46.439933  46.559864  46.640782  46.705092  46.809812  46.899291   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  32.647187  32.782157  32.926560  33.095559  33.199344  33.329930   
1  35.684249  35.738043  35.795595  35.862951  35.904315  35.956361   
2  33.914063  34.118675  34.337587  34.593785  34.751122  34.949087   
3  35.874304  35.924156  35.977491  36.039911  36.078244  36.126476   
4  46.982925  47.066559  47.156038  47.260758  47.325068  47.405986   

        0.95  Actual  
0  33.523478    34.0  
1  36.033501    36.0  
2  35.242502    36.0  
3  36.197963    35.0  
4  47.525918    44.6  


In [39]:
quantile_regression_print(predictions_NGB_df,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

First few actual values: [34.  36.  36.  35.  44.6]
Coverage of 90% prediction interval: 32.63%


In [40]:
evaluation_features_print(predictions_NGB_df,x_test,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

In [41]:
evaluate_uncertainity_matrix(predictions_NGB_df, quantiles,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/NGBoost.xlsx", model_name='NGBoost')

Quantile crossing before sorting: 0 of 95 rows (0.0%)


{'empirical_quantiles': [np.float64(0.30526315789473685),
  np.float64(0.3473684210526316),
  np.float64(0.3684210526315789),
  np.float64(0.4105263157894737),
  np.float64(0.42105263157894735),
  np.float64(0.43157894736842106),
  np.float64(0.4631578947368421),
  np.float64(0.47368421052631576),
  np.float64(0.5157894736842106),
  np.float64(0.5789473684210527),
  np.float64(0.5789473684210527),
  np.float64(0.6),
  np.float64(0.631578947368421)],
 'average_interval_lengths': [np.float64(0.9661878653938487),
  np.float64(1.4712313830879697),
  np.float64(1.88830503712795)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(57.040113244769266),
 'average_crps': np.float64(1.8960580487402798),
 'nll_values': array([ 3.51328945e+00,  4.75022391e-01,  4.04065841e+00,  9.16453271e+00,
         2.58636248e+01, -2.10278765e-01,  1.00948769e+03,  1.67447559e-01,
         8.80958662e+00,  1.14054135e-01,  6.12734358e-01, -1.17030418e-01,
         2.93869546e+00,  7.13780868e+00,

In [42]:
calibrate_and_plot_intervals(predictions_NGB_df,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

The empirical coverage before calibration is: 33.33%
The empirical coverage after calibration is: 89.58%


In [43]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_NGB_df,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

# **Quantile Regression with Gradient Boosting**

In [44]:
best_params = get_best_model_params(best_scores_autosampler, 'GradientBoosting')


# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_GBR_df = pd.DataFrame()

# Custom quantile loss function
def quantile_loss(q, y_true, y_pred):
    e = y_true - y_pred
    return np.mean(np.maximum(q * e, (q - 1) * e))

# Train a model for each quantile and make predictions using GradientBoostingRegressor
for q in quantiles:

    params = best_params.copy()

    params.update({
        'loss': 'quantile',
        'alpha': q,
        'random_state': 42
    })
    # Initialize the model with quantile-specific settings
    model = GradientBoostingRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_GBR_df[q] = predictions

# Add actual target values to the DataFrame
predictions_GBR_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_GBR_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  27.558429  28.158034  28.695237  29.759350  31.797224  32.751821   
1  27.803930  31.881652  32.999937  30.901353  35.072196  36.709827   
2  31.228235  34.261558  34.919103  32.834624  33.571033  33.364127   
3  28.016282  32.899583  32.922008  30.797593  34.435720  37.346564   
4  32.100973  43.111403  47.811135  48.689268  47.523769  43.981411   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  33.001051  32.606835  33.012777  34.456383  35.578158  35.848034   
1  36.833603  38.275996  35.472062  37.684436  37.586971  36.299256   
2  35.039116  34.709390  34.882271  34.994777  36.552341  38.255700   
3  37.809615  37.443571  35.291020  38.344516  37.325342  36.748463   
4  50.784603  52.458644  48.322664  49.084721  50.024525  57.133185   

        0.95  Actual  
0  38.254421    34.0  
1  38.344373    36.0  
2  39.784630    36.0  
3  38.380881    35.0  
4  60.660625    44.6  


In [45]:
quantile_regression_print(predictions_GBR_df,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

First few actual values: [34.  36.  36.  35.  44.6]
Coverage of 90% prediction interval: 81.05%


In [46]:
evaluation_features_print(predictions_GBR_df,x_test,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

In [47]:
evaluate_uncertainity_matrix(predictions_GBR_df, quantiles,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx", model_name='GBM')

Quantile crossing before sorting: 95 of 95 rows (100.0%)


{'empirical_quantiles': [np.float64(0.07368421052631578),
  np.float64(0.15789473684210525),
  np.float64(0.23157894736842105),
  np.float64(0.2631578947368421),
  np.float64(0.3473684210526316),
  np.float64(0.42105263157894735),
  np.float64(0.5052631578947369),
  np.float64(0.5578947368421052),
  np.float64(0.5894736842105263),
  np.float64(0.7052631578947368),
  np.float64(0.7473684210526316),
  np.float64(0.8),
  np.float64(0.8842105263157894)],
 'average_interval_lengths': [np.float64(3.3846915281401224),
  np.float64(7.2130645158007685),
  np.float64(12.542768547664677)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(2.475713432782444),
 'average_crps': np.float64(1.844495233590793),
 'nll_values': array([2.17169693, 2.08772123, 1.96715599, 2.22052601, 3.19107019,
        1.65418232, 3.35665383, 1.57379239, 3.28022642, 2.27508214,
        1.27742377, 2.17905892, 1.75364256, 2.70138876, 3.13218498,
        1.80280681, 2.80997276, 2.0688285 , 2.39985937, 1.418893

In [48]:
calibrate_and_plot_intervals(predictions_GBR_df,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

The empirical coverage before calibration is: 81.25%
The empirical coverage after calibration is: 83.33%


In [49]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_GBR_df,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

# **Quantile Regression with CatBoost**

In [50]:
# Get the best hyperparameters for CatBoostRegressor
best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_CatBoost_df = pd.DataFrame()

# Parameters that may conflict with quantile-specific settings
incompatible_keys = ['objective', 'loss_function', 'eval_metric']

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Remove incompatible parameters
    for key in incompatible_keys:
        params.pop(key, None)

    # Update params with quantile-specific settings
    params.update({
        'loss_function': 'Quantile:alpha={}'.format(q),
        'random_seed': 42
    })

    # Initialize the model with the updated parameters
    model = CatBoostRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train, verbose=0)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_CatBoost_df[q] = predictions

# Add actual target values to the DataFrame
predictions_CatBoost_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_CatBoost_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  24.669272  25.710957  28.375180  29.054732  32.678053  32.977499   
1  32.320548  35.056012  34.739139  35.732348  35.663811  36.927784   
2  28.419844  31.109360  33.805723  33.963950  34.922540  34.554896   
3  29.791827  33.789303  34.537126  35.779426  35.746287  37.172997   
4  46.829837  43.048938  45.520507  44.614015  45.128911  43.434440   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  33.000547  33.047980  32.957970  34.938559  36.482179  37.864789   
1  36.710862  36.678746  36.365674  36.869146  36.510403  38.095942   
2  34.990191  34.523375  34.911108  36.513922  37.401893  40.676569   
3  37.691872  36.604768  37.150379  36.738388  37.847201  38.682597   
4  45.267883  41.629561  41.347857  40.905609  39.392454  43.453697   

        0.95  Actual  
0  39.864972    34.0  
1  37.478155    36.0  
2  41.889188    36.0  
3  40.754275    35.0  
4  43.748389    44.6  


In [51]:
quantile_regression_print(predictions_CatBoost_df,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

First few actual values: [34.  36.  36.  35.  44.6]
Coverage of 90% prediction interval: 81.05%


In [52]:
evaluation_features_print(predictions_CatBoost_df,x_test,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

In [53]:
evaluate_uncertainity_matrix(predictions_CatBoost_df, quantiles,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/CatBoost.xlsx", model_name='CatBoost')

Quantile crossing before sorting: 93 of 95 rows (97.89%)


{'empirical_quantiles': [np.float64(0.07368421052631578),
  np.float64(0.12631578947368421),
  np.float64(0.18947368421052632),
  np.float64(0.30526315789473685),
  np.float64(0.35789473684210527),
  np.float64(0.4105263157894737),
  np.float64(0.4842105263157895),
  np.float64(0.5789473684210527),
  np.float64(0.6421052631578947),
  np.float64(0.7263157894736842),
  np.float64(0.7684210526315789),
  np.float64(0.8210526315789474),
  np.float64(0.8842105263157894)],
 'average_interval_lengths': [np.float64(3.5556443361489802),
  np.float64(7.372686635360426),
  np.float64(10.629939553259314)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(5852.153661222712),
 'average_crps': np.float64(1.8121360562441615),
 'nll_values': array([ 2.47365271e+00,  1.52400816e+00,  2.36392027e+00,  2.25868579e+00,
         1.86320048e+00, -1.21043492e+00,  5.94720483e+00,  1.27634512e+00,
         3.19397233e+00,  2.46379198e+00,  1.20706014e+00,  7.86297693e-01,
         2.10213108e+00,

In [54]:
calibrate_and_plot_intervals(predictions_CatBoost_df,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

The empirical coverage before calibration is: 79.17%
The empirical coverage after calibration is: 91.67%


In [55]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_CatBoost_df,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

# **Quantile Regression with HistGradientBoosting**

In [56]:

# Assume best_params is a dictionary containing the best parameters for HistGradientBoostingRegressor
best_params = get_best_model_params(best_scores_autosampler, 'HGBR')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_HGBR_df = pd.DataFrame()

# Train a model for each quantile and make predictions using HistGradientBoostingRegressor
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Update params with quantile-specific settings
    params.update({
        'loss': 'quantile',
        'quantile': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = HistGradientBoostingRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_HGBR_df[q] = predictions

# Add actual target values to the DataFrame
predictions_HGBR_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_HGBR_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  27.193956  30.504224  31.073493  30.140973  30.879057  32.702751   
1  30.216906  33.070806  32.102819  32.840583  35.028220  35.420621   
2  33.783762  31.848521  34.298583  33.472334  32.889506  34.970385   
3  30.700291  32.865460  32.009053  32.531460  35.333178  35.280936   
4  36.397349  40.750787  42.683536  41.703333  40.764180  41.263722   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  32.904704  32.824103  32.776838  33.451978  33.087752  35.323567   
1  37.271954  36.826176  38.133742  37.400114  38.138183  38.209686   
2  34.952055  34.820785  34.870759  34.998186  35.078085  37.326886   
3  36.700021  36.376630  38.205131  37.524364  38.642413  38.964906   
4  44.890715  44.542442  45.460385  51.877957  50.550388  50.157459   

        0.95  Actual  
0  38.117194    34.0  
1  39.564026    36.0  
2  36.783945    36.0  
3  39.442806    35.0  
4  54.086340    44.6  


In [57]:
quantile_regression_print(predictions_HGBR_df,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/HGBM.xlsx")

First few actual values: [34.  36.  36.  35.  44.6]
Coverage of 90% prediction interval: 77.89%


In [58]:
evaluation_features_print(predictions_HGBR_df,x_test,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/HGBM.xlsx")

In [59]:
evaluate_uncertainity_matrix(predictions_HGBR_df, quantiles,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/HGBM.xlsx", model_name='HGBM')

Quantile crossing before sorting: 95 of 95 rows (100.0%)


{'empirical_quantiles': [np.float64(0.09473684210526316),
  np.float64(0.1368421052631579),
  np.float64(0.22105263157894736),
  np.float64(0.28421052631578947),
  np.float64(0.35789473684210527),
  np.float64(0.4105263157894737),
  np.float64(0.45263157894736844),
  np.float64(0.5578947368421052),
  np.float64(0.6210526315789474),
  np.float64(0.6842105263157895),
  np.float64(0.7157894736842105),
  np.float64(0.8421052631578947),
  np.float64(0.8736842105263158)],
 'average_interval_lengths': [np.float64(3.336051673101651),
  np.float64(7.899423666168205),
  np.float64(11.338932530958568)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(2.4902304856109003),
 'average_crps': np.float64(1.8312940272426586),
 'nll_values': array([2.1868824 , 2.00548219, 1.65885559, 2.03050489, 2.60113973,
        1.62841074, 3.40329711, 2.36108378, 3.09899699, 2.32724717,
        1.33995964, 1.84805363, 2.40447831, 3.54564869, 3.41077627,
        1.67759149, 3.10448418, 1.83404179, 2.42

In [60]:
calibrate_and_plot_intervals(predictions_HGBR_df, "./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/HGBM.xlsx")

The empirical coverage before calibration is: 75.00%
The empirical coverage after calibration is: 87.50%


In [61]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_HGBR_df, "./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/HGBM.xlsx")

# **Quantile Regression with TabNet**

In [62]:
best_params = get_best_model_params(best_scores_autosampler, 'TabNet')

# Quantiles
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Storage
models_quantile = {}
predictions_TabNet_df = pd.DataFrame()

# Convert data to numpy (TabNet requirement)
X_train_np = np.array(X_train)
y_train_np = np.array(y_train).reshape(-1, 1)
X_test_np  = np.array(X_test)

# Quantile loss (pinball loss)
def quantile_loss(q):
    def loss(y_pred, y_true):
        error = y_true - y_pred
        return torch.mean(torch.maximum(q * error, (q - 1) * error))
    return loss

# FIX Q3: early stopping on a 20% validation split of TRAIN, scored with the pinball loss of
# the quantile being trained (was: eval_set = the training data itself, scored with MSE,
# so early stopping never triggered).
from pytorch_tabnet.metrics import Metric
X_fit_np, X_val_np, y_fit_np, y_val_np = train_test_split(X_train_np, y_train_np, test_size=0.2, random_state=42)

def pinball_metric(q):
    class Pinball(Metric):
        def __init__(self):
            self._name = f'pinball_{q}'
            self._maximize = False
        def __call__(self, y_true, y_score, weights=None):  # FIX B8: pytorch-tabnet2 also passes sample weights
            e = np.asarray(y_true, dtype=float) - np.asarray(y_score, dtype=float)
            return float(np.mean(np.maximum(q * e, (q - 1) * e)))
    return Pinball

# Train one TabNet per quantile
for q in quantiles:
    model = TabNetRegressor(
        **best_params,
    )

    model.fit(
        X_fit_np,
        y_fit_np,
        eval_set=[(X_val_np, y_val_np)],
        eval_metric=[pinball_metric(q)],
        loss_fn=quantile_loss(q),
        max_epochs=200,
        patience=30,
        batch_size=1024,
        virtual_batch_size=128,
        num_workers=0,
        drop_last=False
    )

    # Store model
    models_quantile[q] = model

    # Predict
    preds = model.predict(X_test_np).ravel()
    predictions_TabNet_df[q] = preds

# Add actual values
predictions_TabNet_df['Actual'] = y_test.values.ravel()

print(predictions_TabNet_df.head())


Early stopping occurred at epoch 92 with best_epoch = 62 and best_val_0_pinball_0.05 = 0.3615



Early stopping occurred at epoch 91 with best_epoch = 61 and best_val_0_pinball_0.1 = 0.86497



Early stopping occurred at epoch 71 with best_epoch = 41 and best_val_0_pinball_0.15 = 1.16359



Early stopping occurred at epoch 71 with best_epoch = 41 and best_val_0_pinball_0.2 = 1.39967



Early stopping occurred at epoch 135 with best_epoch = 105 and best_val_0_pinball_0.3 = 1.72201



Early stopping occurred at epoch 79 with best_epoch = 49 and best_val_0_pinball_0.4 = 1.61926


... (3 outputs left out)


Early stopping occurred at epoch 60 with best_epoch = 30 and best_val_0_pinball_0.8 = 1.63904



Early stopping occurred at epoch 81 with best_epoch = 51 and best_val_0_pinball_0.85 = 1.21748



Early stopping occurred at epoch 135 with best_epoch = 105 and best_val_0_pinball_0.9 = 0.788



Early stopping occurred at epoch 90 with best_epoch = 60 and best_val_0_pinball_0.95 = 0.35968
        0.05        0.1       0.15        0.2        0.3        0.4  \
0  19.603386  19.859352  20.854452  21.780552  24.816269  27.717943   
1  23.754356  25.781094  25.367247  29.847742  32.481087  25.112562   
2  32.602325  30.927528  30.812885  32.234219  32.676609  35.291622   
3  22.681099  25.216274  24.402542  27.673149  32.991795  29.780470   
4  31.670345  34.696789  34.841904  32.013771  39.848888  46.529922   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  25.918030  34.780140  41.916252  40.556900  35.835480  32.653904   
1  33.963215  35.328850  46.765263  43.048916  41.073845  34.247650   
2  35.658272  35.878296  39.905323  40.450508  39.950859  38.853558   
3  32.454830  39.239952  44.199684  44.313137  42.391045  36.496151   
4  37.433525  50.776215  29.196247  44.408203  41.763481  47.600708   

        0.95  Actual  
0  33.471481    34.0  
1  3

In [63]:
quantile_regression_print(predictions_TabNet_df,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/TabNet.xlsx")

First few actual values: [34.  36.  36.  35.  44.6]
Coverage of 90% prediction interval: 95.79%


In [64]:
evaluation_features_print(predictions_TabNet_df,x_test,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/TabNet.xlsx")

In [65]:
evaluate_uncertainity_matrix(predictions_TabNet_df, quantiles, "./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/TabNet.xlsx", model_name='TabNet')

Quantile crossing before sorting: 95 of 95 rows (100.0%)


{'empirical_quantiles': [np.float64(0.0),
  np.float64(0.010526315789473684),
  np.float64(0.09473684210526316),
  np.float64(0.16842105263157894),
  np.float64(0.23157894736842105),
  np.float64(0.29473684210526313),
  np.float64(0.3263157894736842),
  np.float64(0.4105263157894737),
  np.float64(0.5894736842105263),
  np.float64(0.7473684210526316),
  np.float64(0.8421052631578947),
  np.float64(0.8947368421052632),
  np.float64(0.9578947368421052)],
 'average_interval_lengths': [np.float64(7.672629527041787),
  np.float64(14.2024389166581),
  np.float64(19.185020908556485)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(3.010354887658281),
 'average_crps': np.float64(2.695662692331816),
 'nll_values': array([3.26222462, 2.90650254, 2.00061753, 2.84894976, 3.06219037,
        2.79376012, 4.40854757, 2.70031243, 3.58440369, 2.77657375,
        2.6807949 , 2.64173455, 2.67063634, 2.64402219, 2.69506856,
        3.01027547, 4.91144525, 2.23353247, 3.29986534, 2.7383195

In [66]:
calibrate_and_plot_intervals(predictions_TabNet_df, "./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/TabNet.xlsx")

The empirical coverage before calibration is: 97.92%
The empirical coverage after calibration is: 97.92%


In [67]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_TabNet_df,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/TabNet.xlsx")

# **Quantile Regression with PGBM**

In [68]:
class PGBMWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, **params):
        self.params = params
        self.model = None

    def fit(self, X, y):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        y_ = y.to_numpy() if hasattr(y, "to_numpy") else np.array(y)
        self.model = PGBM()
        self.model.train(
            train_set=(X_, y_),
            objective=mseloss_objective,
            metric=rmseloss_metric,
            params=self.params
        )
        return self

    def predict(self, X):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        return self.model.predict(X_).numpy()


In [69]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'PGBM')

# Remove any conflicting parameters from best_params
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize PGBM model
pgbm_model = PGBM()
def mseloss_objective(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

# Fit the model
pgbm_model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params=best_params)

# Predict the distribution
pred_dist = pgbm_model.predict_dist(X_test)

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# DataFrame to store predictions
predictions_PGBM_df = pd.DataFrame()

# Calculate and store quantiles
for q in quantiles:
    predictions_PGBM_df[q] = np.quantile(pred_dist, q, axis=0)

# Add actual target values to the DataFrame
predictions_PGBM_df['Actual'] = y_test.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_PGBM_df.head())

Training on CPU
Estimator 0/300, Train metric: 9.3540


Estimator 1/300, Train metric: 8.7014
Estimator 2/300, Train metric: 8.2176
Estimator 3/300, Train metric: 7.7928
Estimator 4/300, Train metric: 7.4588
Estimator 5/300, Train metric: 7.0766
Estimator 6/300, Train metric: 6.7824
Estimator 7/300, Train metric: 6.3936
Estimator 8/300, Train metric: 6.0929
Estimator 9/300, Train metric: 5.8563
Estimator 10/300, Train metric: 5.5902
Estimator 11/300, Train metric: 5.3150
Estimator 12/300, Train metric: 5.1076
Estimator 13/300, Train metric: 4.9502
Estimator 14/300, Train metric: 4.7788
Estimator 15/300, Train metric: 4.6107
Estimator 16/300, Train metric: 4.4217
Estimator 17/300, Train metric: 4.2602
Estimator 18/300, Train metric: 4.0980


Estimator 19/300, Train metric: 3.9577
Estimator 20/300, Train metric: 3.8037
Estimator 21/300, Train metric: 3.6834
Estimator 22/300, Train metric: 3.5716
Estimator 23/300, Train metric: 3.4367
Estimator 24/300, Train metric: 3.3685
Estimator 25/300, Train metric: 3.2836
Estimator 26/300, Train metric: 3.1784
Estimator 27/300, Train metric: 3.1301
Estimator 28/300, Train metric: 3.0346
Estimator 29/300, Train metric: 2.9881
Estimator 30/300, Train metric: 2.9410
Estimator 31/300, Train metric: 2.8818
Estimator 32/300, Train metric: 2.8382
Estimator 33/300, Train metric: 2.7729
Estimator 34/300, Train metric: 2.7350
Estimator 35/300, Train metric: 2.6757
Estimator 36/300, Train metric: 2.6177


Estimator 37/300, Train metric: 2.5424
Estimator 38/300, Train metric: 2.5075
Estimator 39/300, Train metric: 2.4696
Estimator 40/300, Train metric: 2.4056
Estimator 41/300, Train metric: 2.3744
Estimator 42/300, Train metric: 2.3309
Estimator 43/300, Train metric: 2.2890
Estimator 44/300, Train metric: 2.2685
Estimator 45/300, Train metric: 2.2192
Estimator 46/300, Train metric: 2.2087
Estimator 47/300, Train metric: 2.1725
Estimator 48/300, Train metric: 2.1472
Estimator 49/300, Train metric: 2.1308
Estimator 50/300, Train metric: 2.0869
Estimator 51/300, Train metric: 2.0626
Estimator 52/300, Train metric: 2.0371
Estimator 53/300, Train metric: 2.0172
Estimator 54/300, Train metric: 1.9750


Estimator 55/300, Train metric: 1.9376
Estimator 56/300, Train metric: 1.9247
Estimator 57/300, Train metric: 1.8948
Estimator 58/300, Train metric: 1.8667
Estimator 59/300, Train metric: 1.8407
Estimator 60/300, Train metric: 1.8067
Estimator 61/300, Train metric: 1.7990
Estimator 62/300, Train metric: 1.7828
Estimator 63/300, Train metric: 1.7553
Estimator 64/300, Train metric: 1.7234
Estimator 65/300, Train metric: 1.7172
Estimator 66/300, Train metric: 1.6990
Estimator 67/300, Train metric: 1.6698
Estimator 68/300, Train metric: 1.6585
Estimator 69/300, Train metric: 1.6427
Estimator 70/300, Train metric: 1.6199
Estimator 71/300, Train metric: 1.6148
Estimator 72/300, Train metric: 1.5931


Estimator 73/300, Train metric: 1.5767
Estimator 74/300, Train metric: 1.5664
Estimator 75/300, Train metric: 1.5543
Estimator 76/300, Train metric: 1.5420
Estimator 77/300, Train metric: 1.5259
Estimator 78/300, Train metric: 1.5041
Estimator 79/300, Train metric: 1.4966
Estimator 80/300, Train metric: 1.4889
Estimator 81/300, Train metric: 1.4770
Estimator 82/300, Train metric: 1.4581
Estimator 83/300, Train metric: 1.4455
Estimator 84/300, Train metric: 1.4385
Estimator 85/300, Train metric: 1.4257
Estimator 86/300, Train metric: 1.4140
Estimator 87/300, Train metric: 1.4040
Estimator 88/300, Train metric: 1.3936
Estimator 89/300, Train metric: 1.3728
Estimator 90/300, Train metric: 1.3609


... (9 outputs left out)

Estimator 253/300, Train metric: 0.6970
Estimator 254/300, Train metric: 0.6958
Estimator 255/300, Train metric: 0.6950
Estimator 256/300, Train metric: 0.6937
Estimator 257/300, Train metric: 0.6924
Estimator 258/300, Train metric: 0.6910
Estimator 259/300, Train metric: 0.6915
Estimator 260/300, Train metric: 0.6902
Estimator 261/300, Train metric: 0.6865
Estimator 262/300, Train metric: 0.6858
Estimator 263/300, Train metric: 0.6851
Estimator 264/300, Train metric: 0.6847
Estimator 265/300, Train metric: 0.6830
Estimator 266/300, Train metric: 0.6815
Estimator 267/300, Train metric: 0.6809
Estimator 268/300, Train metric: 0.6813
Estimator 269/300, Train metric: 0.6801
Estimator 270/300, Train metric: 0.6788


Estimator 271/300, Train metric: 0.6774
Estimator 272/300, Train metric: 0.6748
Estimator 273/300, Train metric: 0.6747
Estimator 274/300, Train metric: 0.6742
Estimator 275/300, Train metric: 0.6732
Estimator 276/300, Train metric: 0.6701
Estimator 277/300, Train metric: 0.6680
Estimator 278/300, Train metric: 0.6676
Estimator 279/300, Train metric: 0.6642
Estimator 280/300, Train metric: 0.6630
Estimator 281/300, Train metric: 0.6630
Estimator 282/300, Train metric: 0.6619
Estimator 283/300, Train metric: 0.6608
Estimator 284/300, Train metric: 0.6602
Estimator 285/300, Train metric: 0.6581
Estimator 286/300, Train metric: 0.6562
Estimator 287/300, Train metric: 0.6562
Estimator 288/300, Train metric: 0.6535
Estimator 289/300, Train metric: 0.6510


Estimator 290/300, Train metric: 0.6493
Estimator 291/300, Train metric: 0.6458
Estimator 292/300, Train metric: 0.6446
Estimator 293/300, Train metric: 0.6449
Estimator 294/300, Train metric: 0.6438
Estimator 295/300, Train metric: 0.6426
Estimator 296/300, Train metric: 0.6412
Estimator 297/300, Train metric: 0.6405
Estimator 298/300, Train metric: 0.6394
Estimator 299/300, Train metric: 0.6384


        0.05        0.1       0.15        0.2        0.3        0.4  \
0  32.613903  32.712513  32.806610  32.871300  32.933006  32.968895   
1  36.357594  36.489403  36.528233  36.565876  36.623905  36.670753   
2  34.306561  34.349525  34.520794  34.604797  34.688782  34.740742   
3  36.493179  36.536526  36.618469  36.665985  36.766624  36.820503   
4  42.051926  42.244644  42.344818  42.475914  42.562881  42.680779   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  33.013924  33.078548  33.141739  33.208607  33.240299  33.328312   
1  36.722866  36.785667  36.856281  36.925179  36.957905  37.133049   
2  34.778641  34.845863  34.908909  35.034164  35.110901  35.156506   
3  36.903473  36.965099  37.034515  37.142761  37.214592  37.252678   
4  42.777512  42.944244  43.104347  43.261200  43.377827  43.567608   

        0.95  Actual  
0  33.510792    34.0  
1  37.218525    36.0  
2  35.293854    36.0  
3  37.407856    35.0  
4  43.795647    44.6  


In [70]:
quantile_regression_print(predictions_PGBM_df,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/PGBM.xlsx")

First few actual values: [34.  36.  36.  35.  44.6]
Coverage of 90% prediction interval: 16.84%


In [71]:
evaluation_features_print(predictions_PGBM_df,x_test,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/PGBM.xlsx")

In [72]:
evaluate_uncertainity_matrix(predictions_PGBM_df, quantiles, "./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/PGBM.xlsx", model_name='PGBM')

Quantile crossing before sorting: 0 of 95 rows (0.0%)


{'empirical_quantiles': [np.float64(0.3684210526315789),
  np.float64(0.3684210526315789),
  np.float64(0.3684210526315789),
  np.float64(0.3684210526315789),
  np.float64(0.4),
  np.float64(0.4421052631578947),
  np.float64(0.45263157894736844),
  np.float64(0.4842105263157895),
  np.float64(0.49473684210526314),
  np.float64(0.5157894736842106),
  np.float64(0.5263157894736842),
  np.float64(0.5263157894736842),
  np.float64(0.5368421052631579)],
 'average_interval_lengths': [np.float64(0.4469910671836452),
  np.float64(0.7338451385498047),
  np.float64(1.0494168130975021)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(36.022580664734),
 'average_crps': np.float64(1.9737758799362182),
 'nll_values': array([ 6.16008023e+00,  3.39312316e+00,  7.99628025e+00,  2.30727546e+01,
         6.19514900e+00, -7.35780464e-02,  2.51442315e+02,  2.00428428e+00,
         1.17354536e+01, -3.97142976e-01,  1.15905938e+00, -1.51271007e-01,
         1.38117267e+01,  6.61715241e+01,  

In [73]:
calibrate_and_plot_intervals(predictions_PGBM_df, "./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/PGBM.xlsx")

The empirical coverage before calibration is: 18.75%
The empirical coverage after calibration is: 89.58%


In [74]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_PGBM_df,"./drive/MyDrive/Regression/concrete_uncertainty_analysis/quantile_regression/PGBM.xlsx")